This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [37]:
#!pip install keras keras-hub --upgrade -q

In [38]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [39]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## ConvNet architecture patterns

이 장에서는 다음 내용을 다룹니다.

* 모델 아키텍처를 위한 **모듈성-계층성-재사용** 공식
* 컨볼루션 신경망 구축을 위한 **표준 모범 사례** 개요: residual connections, batch normalization, depthwise separable convolutions
* 컴퓨터 비전 모델의 **최신 설계** 트렌드
---
모델의 "아키텍처"는 모델을 만드는 데 있어 고려된 모든 선택의 총합입니다. 어떤 레이어를 사용할지, 어떻게 구성할지, 어떤 방식으로 연결할지 등이 여기에 포함됩니다. 이러한 선택은 모델의 가설 공간을 정의합니다. 가설 공간이란 경사 하강법이 탐색할 수 있는 가능한 함수의 공간으로, 모델의 가중치에 의해 매개변수화됩니다. 특징 엔지니어링과 마찬가지로, 좋은 가설 공간은 당면한 문제와 그 해결책에 대한 사전 지식을 반영합니다. 예를 들어, **컨볼루션 레이어를 사용한다는 것은 입력 이미지에 존재하는 관련 패턴이 이동 불변(translation-invariant)이라는 것을 미리 알고 있다는 의미**입니다. 데이터를 효과적으로 학습하려면 무엇을 찾고자 하는지에 대한 가정을 세워야 합니다.

모델 아키텍처는 모델의 성공과 실패를 가르는 중요한 요소입니다. 부적절한 아키텍처를 선택하면 모델의 성능이 최적화되지 못하고, 아무리 많은 훈련 데이터를 사용해도 개선되지 않을 수 있습니다. 반대로, 좋은 모델 아키텍처는 학습 속도를 높이고 모델이 사용 가능한 훈련 데이터를 효율적으로 활용할 수 있도록 하여 대규모 데이터셋의 필요성을 줄여줍니다. 좋은 모델 아키텍처란 탐색 공간의 크기를 줄이거나, 탐색 공간의 좋은 지점으로 수렴하기 쉽게 만드는 아키텍처입니다. 특징 엔지니어링이나 데이터 큐레이션처럼, 모델 아키텍처는 경사 하강법이 문제를 더 쉽게 풀 수 있도록 설계하는 데 중점을 둡니다. 경사 하강법은 매우 단순한 탐색 과정이기 때문에, 가능한 모든 도움을 필요로 합니다.

모델 아키텍처는 과학이라기보다는 예술에 가깝습니다. 숙련된 머신러닝 엔지니어는 직관적으로 처음 시도에 고성능 모델을 만들어낼 수 있는 반면, 초보자는 학습조차 되지 않는 모델을 만드는 데 어려움을 겪는 경우가 많습니다. 여기서 핵심은 '직관'입니다. 무엇이 효과가 있고 무엇이 효과가 없는지 명확하게 설명할 수 있는 사람은 아무도 없습니다. 전문가들은 풍부한 실무 경험을 통해 얻은 패턴 인식 능력을 활용합니다. 이 책을 읽는 동안 여러분도 자신만의 직관을 키워나가게 될 것입니다. 하지만 직관에만 의존하는 것은 아닙니다. 실제 과학적인 근거는 많지 않지만, 다른 공학 분야와 마찬가지로 모범 사례는 존재합니다.

다음 섹션에서는 몇 가지 필수적인 ConvNet 아키텍처 모범 사례, 특히 잔여 연결(residual connections), 배치 정규화(batch normalization), 분리형 컨볼루션(separable convolution)에 대해 살펴보겠습니다. 이러한 기법들을 숙달하면 매우 효과적인 이미지 모델을 구축할 수 있습니다. 개와 고양이를 분류하는 문제에 이러한 기법들을 적용하는 방법을 보여드리겠습니다.

먼저 시스템 아키텍처의 전체적인 관점, 즉 모듈성-계층성-재사용(MHR) 공식부터 살펴보겠습니다.

### Modularity, hierarchy, and reuse

복잡한 시스템을 단순화하고 싶다면 적용할 수 있는 보편적인 방법이 있습니다. 바로 **복잡한 시스템을 모듈로 구조화**하고, **모듈들을 계층 구조로 구성**한 다음, 필요에 따라 **여러 곳에서 동일한 모듈을 재사용**하는 것입니다("재사용"은 추상화의 다른 표현입니다). 이것이 바로 모듈성-계층 구조-재사용(MHR) 공식(그림 9.1 참조)이며, 아키텍처라는 용어가 사용되는 거의 모든 영역의 시스템 아키텍처의 기본 원리입니다. 성당이든, 인체든, 미 해군이든, 케라스 코드베이스든, 의미 있는 복잡성을 지닌 모든 시스템의 조직화에 있어 핵심적인 역할을 합니다.

<img src="https://deeplearningwithpython.io/images/ch09/complex_systems.e89aaf78.png" width="600"><p style="text-align:center">Figure 9.1: Complex systems follow a hierarchical structure and are organized into distinct modules, which are reused multiple times (such as your 4 limbs, which are all variants of the same blueprint, or your 20 fingers).</p>

소프트웨어 엔지니어라면 이미 이러한 원칙들을 잘 알고 있을 것입니다. 효율적인 코드베이스란 모듈화되고 계층적이며, 동일한 기능을 두 번 다시 구현하지 않고 재사용 가능한 클래스와 함수를 활용하는 구조입니다. 이러한 원칙들을 따라 코드를 팩터링하는 것을 "소프트웨어 아키텍처"라고 부를 수 있습니다.

딥러닝 자체는 바로 이 원칙을 경사 하강법을 이용한 연속 최적화에 적용한 것입니다. 고전적인 최적화 기법(연속 함수 공간에서의 경사 하강법)을 사용하여 탐색 공간을 모듈(레이어)로 구성하고, 이 모듈들을 깊은 계층 구조(가장 간단한 형태인 스택)로 배열하여 재사용 가능한 모든 것을 재사용하는 것입니다(예를 들어, 컨볼루션은 서로 다른 공간 위치에 있는 동일한 정보를 재사용하는 것입니다).

마찬가지로, 딥러닝 모델 아키텍처는 모듈성, 계층 구조, 그리고 재사용을 영리하게 활용하는 것이 핵심입니다. 널리 알려진 컨볼루션 신경망(ConvNet) 아키텍처들은 단순히 레이어로만 구성된 것이 아니라, 레이어들이 반복되는 그룹(블록 또는 모듈)으로 구성되어 있다는 것을 알 수 있습니다. 예를 들어, 이전 장에서 사용한 Xception 아키텍처는 SeparableConv - SeparableConv - MaxPooling 블록이 반복되는 구조로 되어 있습니다(그림 9.2 참조).

또한, 대부분의 컨볼루션 신경망은 피라미드 형태의 구조(특징 계층 구조)를 갖는 경우가 많습니다. 이전 장에서 구축한 첫 번째 컨볼루션 신경망에서 사용한 컨볼루션 필터 개수의 증가 추세를 떠올려 보세요: 32개, 64개, 128개. 필터 개수는 레이어 깊이에 따라 증가하는 반면, 특징 맵의 크기는 그에 따라 줄어듭니다. **Xception 모델의 블록**에서도 동일한 패턴을 확인할 수 있습니다(그림 9.2 참조).

<img src="https://deeplearningwithpython.io/images/ch09/xception_entry_flow_pyramid.4701a5de.png" width="600"><p style="text-align:center">Figure 9.2: The “entry flow” of the Xception architecture: note the repeated layer blocks and the gradually shrinking and deepening feature maps, going from 299 x 299 x 3 to 19 x 19 x 728.</p>

더 깊은 계층 구조는 본질적으로 기능 재사용을 촉진하고 결과적으로 추상화를 용이하게 하기 때문에 좋습니다. **일반적으로 좁은 계층을 깊게 쌓은 구조가 넓은 계층을 얕게 쌓은 구조보다 성능이 더 좋습니다.** 하지만 계층을 **깊게 쌓는 데에는 한계**가 있는데, 바로 **기울기 소실 문제**입니다. 이 때문에 우리는 첫 번째 필수 모델 아키텍처 패턴인 **잔차 연결(residual connections**)을 살펴보게 됩니다.

```
딥러닝 연구에서 어블레이션 연구의 중요성

딥러닝 아키텍처는 설계된 것보다 진화된 경우가 많습니다. 즉, 여러 번 시도하고 효과가 있는 것처럼 보이는 것들을 선택하는 과정을 통해 개발됩니다. 생물학적 시스템과 마찬가지로, 복잡한 실험적 딥러닝 시스템을 살펴보면, 성능 저하 없이 몇몇 모듈을 제거하거나 학습된 특징들을 무작위로 대체할 수 있는 경우가 많습니다.

이러한 현상은 딥러닝 연구자들이 직면하는 인센티브 때문에 더욱 심화됩니다. 시스템을 필요 이상으로 복잡하게 만들면 더 흥미롭거나 참신해 보여 논문 심사를 통과할 가능성이 높아지기 때문입니다. 많은 딥러닝 논문을 읽어보면, 논문의 스타일과 내용 모두에서 동료 심사에 최적화되어 있어 설명의 명확성과 결과의 신뢰성을 오히려 저해하는 경우가 많다는 것을 알 수 있습니다. 예를 들어, 딥러닝 논문에서 수학은 개념을 명확하게 형식화하거나 예상치 못한 결과를 도출하는 데 사용되기보다는, 마치 영업사원이 값비싼 정장을 입는 것처럼 연구의 진지함을 과시하는 수단으로 사용되는 경우가 많습니다.

연구의 목표는 단순히 논문을 발표하는 것이 아니라 신뢰할 수 있는 지식을 창출하는 것이어야 합니다. 무엇보다 중요한 것은 시스템 내 인과관계를 이해하는 것이 신뢰할 수 있는 지식을 생성하는 가장 직접적인 방법이라는 점입니다. 그리고 인과관계를 살펴보는 데에는 아주 적은 노력이 필요한 방법이 있는데, 바로 어블레이션 연구(ablation study)입니다. 어블레이션 연구는 시스템의 일부를 체계적으로 제거해 보는, 즉 시스템을 단순화해 보는 연구로, 시스템 성능의 실제 원인이 무엇인지 파악하는 데 도움이 됩니다. 만약 X + Y + Z 조합이 좋은 결과를 가져온다면, X, Y, Z, X + Y, X + Z, Y + Z 조합도 시도해 보고 어떤 결과가 나오는지 살펴보세요.

딥러닝 연구자가 된다면, 연구 과정에서 불필요한 정보를 걸러내고 모델에 대한 어블레이션 연구를 수행하세요. 항상 다음과 같은 질문을 스스로에게 던져보세요. 더 간단한 설명은 없을까? 이 추가적인 복잡성이 정말 필요한 것일까? 왜 필요한 것일까?
```

### Residual connections

영국에서는 '중국 속삭임', 프랑스에서는 '텔레폰 아랍'이라고도 불리는 '전화 게임'을 아실 겁니다. 이 게임은 한 플레이어의 귀에 처음 메시지를 속삭이고, 그 플레이어가 다음 플레이어의 귀에 속삭이는 식으로 계속 이어집니다. 결국 최종 메시지는 원래 메시지와는 거의 비슷하지 않게 됩니다. 이는 잡음이 많은 채널을 통해 순차적으로 메시지를 전달할 때 발생하는 누적 오류를 재미있게 비유한 것입니다.

실제로 **순차적인 딥러닝 모델에서 역전파는 '전화 게임'과 매우 유사**합니다. 다음과 같은 함수들의 연결 고리가 있습니다.

y = f4(f3(f2(f1(x))))

이 게임의 목표는 함수 f4의 출력에 기록된 오류(모델의 손실)를 기반으로 연결 고리에 있는 각 함수의 매개변수를 조정하는 것입니다. f1을 조정하려면 f2, f3, f4를 통해 오류 정보를 전달해야 합니다. 하지만 연결 고리의 각 함수는 이 과정에 어느 정도의 잡음을 발생시킵니다. **함수 체인이 너무 깊으면 노이즈가 기울기 정보를 압도**하여 역전파가 제대로 작동하지 않게 됩니다. 결국 **모델은 학습되지 않습니다**. 이를 **기울기 소실 문제(vanishing gradients problem**)라고 합니다.

해결 방법은 간단합니다. 체인의 각 함수가 비파괴적(nondisstructive)이 되도록, 즉 **이전 입력에 포함된 정보의 노이즈가 제거된 버전을 유지하도록 강제**하면 됩니다. 이를 구현하는 가장 쉬운 방법은 **잔차 연결(residual connection**)이라고 합니다. 매우 간단합니다. 레이어 또는 레이어 블록의 입력을 출력에 다시 연결하면 됩니다(그림 9.3 참조). 잔차 연결은 파괴적이거나 노이즈가 많은 블록(예: ReLU 활성화 함수 또는 드롭아웃 레이어가 포함된 블록)을 우회하는 정보 지름길 역할을 하여 초기 레이어의 오류 기울기 정보가 깊은 네트워크를 통해 노이즈 없이 전파될 수 있도록 합니다. 이 기술은 **2015년 Microsoft의 He 등이 개발한 ResNet 모델**군에서 처음 소개되었습니다.[1]

<img src="https://deeplearningwithpython.io/images/ch09/residual_connection.0524fdc4.png" width="200"><p style="text-align:center">Figure 9.3: A residual connection around a processing block</p>

In [40]:
import keras
from keras import layers

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
residual = layers.Conv2D(64, 1)(residual)
x = layers.add([x, residual])

**블록의 출력에 입력을 다시 더한다는 것은 출력이 입력과 동일한 형태를 가져야 함을 의미**합니다. 하지만 블록에 필터 수가 증가된 컨볼루션 레이어나 맥스 풀링 레이어가 포함된 경우에는 그렇지 않습니다. 이러한 경우에는 **활성화 함수가 없는 1×1 Conv2D 레이어를 사용하여 잔차를 원하는 출력 형태로 선형 투영**합니다. 일반적으로 대상 블록의 컨볼루션 레이어에서 **패딩으로 인한 공간적 다운샘플링을 방지하기 위해 padding="same"을 사용**하고, **맥스 풀링 레이어로 인한 다운샘플링을 보정하기 위해 잔차 투영에 strides를 사용**합니다.

In [41]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)
residual = layers.Conv2D(64, 1, strides=2)(residual)
x = layers.add([x, residual])

이러한 개념을 보다 구체적으로 설명하기 위해, **두 개의 컨볼루션 레이어와 하나의 선택적 맥스 풀링 레이어로 구성된 일련의 블록**으로 이루어진 간단한 컨볼루션 신경망(ConvNet)의 예를 제시합니다. **각 블록은 잔차 연결(residual connection)로 둘러싸여** 있습니다.

In [42]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Rescaling(1.0 / 255)(inputs)

def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    if pooling:
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(filters, 1, strides=2)(residual)
    elif filters != residual.shape[-1]:
        residual = layers.Conv2D(filters, 1)(residual)
    x = layers.add([x, residual])
    return x

x = residual_block(x, filters=32, pooling=True)
x = residual_block(x, filters=64, pooling=True)
x = residual_block(x, filters=128, pooling=False)

x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

모델 요약을 살펴보겠습니다.

In [43]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)   │ (None, 32, 32, 3)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_3 (Rescaling)       │ (None, 32, 32, 3)         │               0 │ input_layer_11[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_52 (Conv2D)            │ (None, 32, 32, 32)        │             896 │ rescaling_3[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_53 (Conv2D)            │ (None, 32, 32, 32)        │           9,248 │ conv2d_52[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_10              │ (None, 16, 16, 32)        │               0 │ conv2d_53[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_54 (Conv2D)            │ (None, 16, 16, 32)        │             128 │ rescaling_3[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_17 (Add)                  │ (None, 16, 16, 32)        │               0 │ max_pooling2d_10[0][0],    │
│                               │                           │                 │ conv2d_54[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_55 (Conv2D)            │ (None, 16, 16, 64)        │          18,496 │ add_17[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_56 (Conv2D)            │ (None, 16, 16, 64)        │          36,928 │ conv2d_55[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_11              │ (None, 8, 8, 64)          │               0 │ conv2d_56[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_57 (Conv2D)            │ (None, 8, 8, 64)          │           2,112 │ add_17[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_18 (Add)                  │ (None, 8, 8, 64)          │               0 │ max_pooling2d_11[0][0],    │
│                               │                           │                 │ conv2d_57[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_58 (Conv2D)            │ (None, 8, 8, 128)         │          73,856 │ add_18[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_59 (Conv2D)            │ (None, 8, 8, 128)         │         147,584 │ conv2d_58[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_60 (Conv2D)            │ (None, 8, 8, 128)         │           8,320 │ add_18[0][0]               │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 297,697 (1.14 MB)

 Trainable params: 297,697 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

**잔여 연결(residual connection)을 사용하면 기울기 소실 문제를 걱정하지 않고 임의의 깊이를 가진 네트워크를 구축할 수 있습니다**. 이제 다음으로 중요한 컨볼루션 신경망 아키텍처 패턴인 배치 정규화에 대해 알아보겠습니다.

### Batch normalization

머신러닝에서 정규화는 머신러닝 모델이 접하는 여러 샘플을 서로 더 유사하게 만들어 모델의 학습 및 새로운 데이터에 대한 일반화 능력을 향상시키는 광범위한 방법 범주입니다. 가장 **일반적인 데이터 정규화 방법**은 이 책에서 이미 여러 번 다룬 것처럼 **데이터의 평균을 빼서 0을 중심으로 하고, 데이터를 표준편차로 나누어 1의 표준편차를 갖도록 하는 것**입니다. 사실상, 이 방법은 데이터가 정규(또는 가우시안) 분포를 따른다고 가정하고, 이 분포를 중심으로 하고 1의 분산을 갖도록 조정하는 것입니다.

```
normalized_data = (data - np.mean(data, axis=...)) / np.std(data, axis=...)
```

이 책에서 살펴본 이전 예제들은 모델에 입력하기 전에 데이터를 정규화했습니다. 하지만 **네트워크가 수행하는 모든 변환 후에 데이터 정규화가 문제가 될 수** 있습니다. Dense 또는 Conv2D 네트워크에 입력되는 데이터의 평균이 0이고 분산이 1이라고 하더라도, 출력되는 데이터도 항상 동일할 것이라고 보장할 수는 없습니다. 중간 활성화 값을 정규화하면 도움이 될까요?

배치 정규화는 바로 이러한 문제를 해결합니다. 배치 정규화는 2015년 Ioffe와 Szegedy가 도입한 레이어 유형(Keras의 BatchNormalization)입니다.[2] **배치 정규화는 학습 과정에서 평균과 분산이 시간에 따라 변하더라도 데이터를 적응적으로 정규화**할 수 있습니다. **학습 중에는 현재 배치 데이터의 평균과 분산을 사용하여 샘플을 정규화**하고, **추론 단계(대표적인 데이터의 충분한 배치를 확보하지 못한 경우)에서는 학습 중에 사용된 데이터의 배치별 평균과 분산의 지수 이동 평균을 사용**합니다.

이오페와 세게디의 원 논문에서는 배치 정규화가 "내부 공변량 이동을 줄이는" 방식으로 작동한다고 제안했지만, **배치 정규화가 정확히 왜 도움이 되는지는 아직 확실히 밝혀지지 않았습니다**. 다양한 가설이 있지만 확실한 것은 없습니다. 딥러닝 분야에서도 이러한 현상이 흔히 나타납니다. 딥러닝은 정확한 과학이 아니라 끊임없이 변화하는 경험적으로 도출된 엔지니어링 모범 사례들의 집합이며, 신뢰할 수 없는 설명들이 뒤섞여 있습니다. 마치 손에 든 책이 어떤 작업을 하는 방법은 알려주지만 왜 그렇게 하는지는 제대로 설명하지 않는 것처럼 느껴질 때가 있습니다. 이는 우리가 방법은 알지만 이유는 모르기 때문입니다. 확실한 설명이 있을 때는 반드시 언급하려고 노력합니다. 하지만 배치 정규화는 그러한 확실한 설명에 해당하지 않습니다.

실제로 배치 정규화의 주요 효과는 잔차 연결과 유사하게 기울기 전파를 개선하여 더 깊은 네트워크를 구현할 수 있게 해준다는 점입니다. 일부 매우 깊은 네트워크는 여러 개의 배치 정규화 레이어를 포함해야만 학습이 가능합니다. 예를 들어, **배치 정규화는 ResNet50, EfficientNet, Xception과 같이 Keras에 포함된 많은 고급 컨볼루션 신경망 아키텍처에서 널리 사용**됩니다.

BatchNormalization 레이어는 Dense, Conv2D 등 어떤 레이어 뒤에도 사용할 수 있습니다.

```
x = ...
# Because the output of the Conv2D layer gets normalized, the layer
# doesn't need its own bias vector.
x = layers.Conv2D(32, 3, use_bias=False)(x)
x = layers.BatchNormalization()(x)
```

```
Dense와 Conv2D 모두 "바이어스 벡터"라는 학습된 변수를 사용하는데, 이 변수의 목적은 레이어를 순수 선형이 아닌 어파인 함수로 만드는 것입니다. 예를 들어, Conv2D는 개략적으로 y = conv(x, kernel) + bias를 반환하고, Dense는 y = dot(x, kernel) + bias를 반환합니다. 배치 정규화(BatchNormalization)를 사용할 경우 정규화 단계에서 레이어 출력을 0으로 중심화하기 때문에 바이어스 벡터가 더 이상 필요하지 않으며, use_bias=False 옵션을 통해 바이어스 벡터 없이 레이어를 생성할 수 있습니다. 이로 인해 레이어가 약간 더 간결해집니다.
```
중요한 점은, **일반적으로 이전 레이어의 활성화 함수를 배치 정규화 레이어 뒤에 배치하는 것을 권장**합니다(물론 이는 여전히 논쟁의 여지가 있는 주제입니다). 따라서 다음과 같이 하는 대신에

```
x = layers.Conv2D(32, 3, activation="relu")(x)
x = layers.BatchNormalization()(x)
```

실제로는 다음과 같이 하시면 됩니다.

```
# Note the lack of activation here.
x = layers.Conv2D(32, 3, use_bias=False)(x)
x = layers.BatchNormalization()(x)
# We place the activation after the BatchNormalization layer.
x = layers.Activation("relu")(x)
```

직관적으로 생각해 보면, 배치 정규화는 입력값을 0을 중심으로 정렬하는 반면, ReLU 활성화 함수는 0을 기준으로 활성화된 채널을 유지할지 버릴지를 결정합니다. 따라서 활성화 함수 전에 정규화를 수행하면 ReLU 활성화 함수의 활용도를 극대화할 수 있습니다. 하지만 이 순서가 반드시 필수적인 것은 아니므로, 컨볼루션-활성화-배치 정규화 순서로 진행하더라도 모델 학습에는 문제가 없으며, 반드시 더 나쁜 결과가 나오는 것도 아닙니다.
```
배치 정규화에는 여러 가지 특이한 점이 있습니다. 그중 하나는 미세 조정과 관련된 것인데, 배치 정규화 레이어가 포함된 모델을 미세 조정할 때는 해당 레이어를 고정(trainable 속성 값을 False로 설정)해 두는 것이 좋습니다. 그렇지 않으면 배치 정규화 레이어의 내부 평균과 분산이 계속 업데이트되어 주변 Conv2D 레이어에 적용되는 아주 작은 업데이트와 충돌할 수 있습니다.
```
자, 이제 이번 시리즈의 마지막 아키텍처 패턴인 깊이별 분리형 컨볼루션을 살펴보겠습니다.

### Depthwise separable convolutions

**Conv2D를 대체할 수 있는 레이어**가 있는데, 이 레이어를 사용하면 **모델의 크기가 작아지고(학습 가능한 가중치 매개변수 감소), 더 효율적이며(부동 소수점 연산 감소), 성능도 몇 퍼센트 향상**된다는 사실을 알려드린다면 어떠시겠습니까? 바로 **Depthwise Separable Convolution 레이어(Keras의 SeparableConv2D)가 그러한 역할**을 합니다. 이 레이어는 그림 9.4에서 보는 것처럼 **입력의 각 채널에 대해 독립적으로 공간 컨볼루션을 수행한 후, 포인트별 컨볼루션(1×1 컨볼루션)을 통해 출력 채널들을 결합**합니다.
<img src="https://deeplearningwithpython.io/images/ch09/depthwise_separable_conv.5d1929bd.png" width="600"><p style="text-align:center">Figure 9.4: Depthwise separable convolution: a depthwise convolution followed by a pointwise convolution</p>

이는 **공간적 특징 학습과 채널별 특징 학습을 분리**하는 것과 같습니다. 컨볼루션이 이미지의 패턴이 특정 위치에 고정되어 있지 않다는 가정에 기반하는 것과 마찬가지로, **깊이별 분리 컨볼루션(DSP)은 중간 활성화 값의 공간적 위치는 높은 상관관계**를 갖지만, **각 채널은 높은 독립성을 갖는다**는 가정에 기반합니다. 이러한 가정은 심층 신경망이 학습하는 이미지 표현에 일반적으로 적용되므로, 모델이 훈련 데이터를 더욱 효율적으로 활용할 수 있도록 도와주는 유용한 사전 정보 역할을 합니다. 처리해야 할 정보 구조에 대한 사전 정보가 강할수록 모델은 더 나은 성능을 보입니다. 단, 사전 정보가 정확해야 합니다.

**깊이별 분리 컨볼루션은 일반 컨볼루션에 비해 훨씬 적은 파라미터와 계산량을 필요로 하면서도 유사한 표현력을 제공**합니다. 결과적으로 **모델의 크기가 작아지고 수렴 속도가 빨라지며 과적합될 가능성이 줄어**듭니다. 이러한 장점은 특히 제한된 데이터로 작은 모델을 처음부터 학습시킬 때 매우 중요해집니다.

대규모 모델의 경우, 깊이별 분리 가능 컨볼루션은 Keras와 함께 제공되는 고성능 ConvNet인 Xception 아키텍처의 기반입니다. 깊이별 분리 가능 컨볼루션과 Xception에 대한 이론적 근거에 대한 자세한 내용은 "Xception: Deep Learning with Depthwise Separable Convolutions"[3] 논문에서 확인할 수 있습니다.

```
하드웨어, 소프트웨어, 알고리즘의 공진화

3x3 윈도우, 64개의 입력 채널, 64개의 출력 채널을 사용하는 일반적인 컨볼루션 연산을 생각해 보겠습니다. 이 연산은 3 × 3 × 64 × 64 = 36,864개의 학습 가능한 파라미터를 사용하며, 이미지에 적용할 때 이 파라미터 개수에 비례하는 부동 소수점 연산을 수행합니다. 반면, 동일한 깊이별 분리형 컨볼루션을 생각해 보면, 학습 가능한 파라미터는 3 × 3 × 64 + 64 × 64 = 4,672개에 불과하고 부동 소수점 연산 횟수도 비례적으로 줄어듭니다. 이러한 효율성 향상은 필터 개수나 컨볼루션 윈도우 크기가 커질수록 더욱 커집니다.

따라서 깊이별 분리형 컨볼루션이 훨씬 빠를 것이라고 예상할 수 있겠죠? 하지만 그렇지 않습니다. 이러한 알고리즘을 간단한 CUDA나 C++로 구현한다면 이 말이 맞을 수도 있습니다. 실제로 CPU에서 실행할 때는 병렬화된 C++ 구현 덕분에 의미 있는 속도 향상을 볼 수 있습니다. 하지만 실제로는 GPU를 사용하고 있을 가능성이 높으며, GPU에서 실행되는 코드는 단순한 CUDA 구현과는 거리가 멉니다. 바로 cuDNN 커널, 즉 각 머신 명령어 단위까지 극도로 최적화된 코드입니다. NVIDIA 하드웨어에서 cuDNN 컨볼루션은 매일 수많은 엑사플롭스의 연산량을 처리하기 때문에 이 코드를 최적화하는 데 많은 노력을 기울이는 것은 당연합니다. 하지만 이러한 극단적인 미세 최적화의 부작용으로 인해 다른 접근 방식, 심지어 깊이별 분리 컨볼루션처럼 본질적인 장점이 있는 접근 방식조차도 성능 경쟁에서 밀릴 가능성이 매우 낮아집니다.

NVIDIA에 여러 차례 요청했음에도 불구하고, 깊이별 분리형 컨볼루션(DSP)은 일반 컨볼루션만큼의 소프트웨어 및 하드웨어 최적화를 거치지 못했습니다. 그 결과, 깊이별 분리형 컨볼루션은 일반 컨볼루션보다 파라미터와 부동 소수점 연산 횟수가 제곱으로 적음에도 불구하고 여전히 일반 컨볼루션과 비슷한 속도에 그치고 있습니다. 하지만 속도 향상이 없더라도 깊이별 분리형 컨볼루션을 사용하는 것은 여전히 ​​좋은 선택입니다. 파라미터 수가 적다는 것은 과적합 위험을 줄여주고, 채널 간 상관관계가 없다는 가정 덕분에 모델 수렴 속도가 빨라지고 더욱 견고한 표현을 얻을 수 있기 때문입니다.

이러한 사소한 불편함이 다른 상황에서는 넘을 수 없는 벽이 될 수 있습니다. 딥러닝의 전체 하드웨어 및 소프트웨어 생태계가 매우 특정한 알고리즘(특히 역전파를 통해 학습된 컨볼루션 신경망)에 맞춰 미세하게 최적화되어 있기 때문에, 기존 방식을 벗어나는 것은 엄청난 비용을 수반합니다. 경사 하강법 없는 최적화나 스파이킹 신경망 같은 대안 알고리즘을 실험해 보려고 한다면, 아무리 기발하고 효율적인 아이디어라 할지라도 처음 몇 개의 병렬 C++ 또는 CUDA 구현은 기존의 컨볼루션 신경망보다 훨씬 느릴 것입니다. 다른 연구자들이 당신의 방법을 채택하도록 설득하는 것은, 설령 당신의 방법이 훨씬 더 뛰어나다 하더라도, 매우 어려운 일일 것입니다.

현대 딥러닝은 하드웨어, 소프트웨어, 그리고 알고리즘 간의 공진화 과정의 산물이라고 할 수 있습니다. NVIDIA GPU와 CUDA의 등장으로 역전파 학습 기반의 컨볼루션 신경망이 초기에 성공을 거두었고, NVIDIA는 이러한 알고리즘에 최적화된 하드웨어와 소프트웨어를 개발했으며, 이는 결국 이러한 방법론을 연구하는 커뮤니티의 결집으로 이어졌습니다. 이제 다른 길을 찾으려면 전체 생태계를 재설계하는 데 수년이 걸릴 것입니다.
```


### Putting it together: A mini Xception-like model

지금까지 배운 ConvNet 아키텍처 원칙을 다시 한번 정리해 보겠습니다.

* 모델은 일반적으로 **여러 개의 컨볼루션 레이어와 맥스 풀링 레이어로 구성된 반복적인 레이어 블록**으로 이루어져야 합니다.
* **레이어의 필터 수는 공간 특징 맵의 크기가 작아질수록 증가**해야 합니다.
* **깊고 좁은 구조**가 넓고 얕은 구조보다 좋습니다.
* **레이어 블록 주변에 잔여 연결(residual connection)을 도입하면 더 깊은 네트워크를 학습하는 데 도움**이 됩니다.
* **컨볼루션 레이어 후에 배치 정규화 레이어를 추가하는 것이 유용**할 수 있습니다.
* **매개변수 효율성이 더 높은 SeparableConv2D 레이어로 Conv2D 레이어를 대체하는 것이 유용**할 수 있습니다.

이제 이러한 모든 아이디어를 하나의 모델에 통합해 보겠습니다. **이 모델의 아키텍처는 Xception의 축소판과 유사**합니다. 지난 장에서 다룬 개와 고양이 비교 문제에 이 모델을 적용해 보겠습니다. 데이터 로딩과 모델 학습은 8장 8.2절에서 사용했던 것과 동일한 설정을 그대로 사용하되, 모델 정의는 다음과 같은 ConvNet으로 대체하면 됩니다.


In [47]:
'''
import kagglehub

kagglehub.login()
'''

In [48]:
'''
import zipfile

download_path = kagglehub.competition_download("dogs-vs-cats")

with zipfile.ZipFile(download_path + "/train.zip", "r") as zip_ref:
    zip_ref.extractall(".")
'''

'\nimport zipfile\n\ndownload_path = kagglehub.competition_download("dogs-vs-cats")\n\nwith zipfile.ZipFile(download_path + "/train.zip", "r") as zip_ref:\n    zip_ref.extractall(".")\n'

In [49]:
import os, shutil, pathlib
from keras.utils import image_dataset_from_directory

original_dir = pathlib.Path("train")
new_base_dir = pathlib.Path("dogs_vs_cats_small")
'''
def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

make_subset("train", start_index=0, end_index=1000)
make_subset("validation", start_index=1000, end_index=1500)
make_subset("test", start_index=1500, end_index=2500)
'''
batch_size = 64
image_size = (180, 180)
train_dataset = image_dataset_from_directory(
    new_base_dir / "train",
    image_size=image_size,
    batch_size=batch_size,
)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation",
    image_size=image_size,
    batch_size=batch_size,
)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test",
    image_size=image_size,
    batch_size=batch_size,
)


Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [50]:
import tensorflow as tf
from keras import layers

data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
]

def data_augmentation(images, targets):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images, targets

augmented_train_dataset = train_dataset.map(
    data_augmentation, num_parallel_calls=8
)
augmented_train_dataset = augmented_train_dataset.prefetch(tf.data.AUTOTUNE)

In [51]:
import keras

inputs = keras.Input(shape=(180, 180, 3))
x = layers.Rescaling(1.0 / 255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=5, use_bias=False)(x)

for size in [32, 64, 128, 256, 512]:
    residual = x

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    residual = layers.Conv2D(
        size, 1, strides=2, padding="same", use_bias=False
    )(residual)
    x = layers.add([x, residual])

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

In [52]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
history = model.fit(
    augmented_train_dataset,
    epochs=100,
    validation_data=validation_dataset,
)

Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 500s 15s/step - accuracy: 0.5350 - loss: 0.7203 - val_accuracy: 0.5000 - val_loss: 0.6951
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 492s 15s/step - accuracy: 0.5990 - loss: 0.6530 - val_accuracy: 0.5000 - val_loss: 0.6933
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 490s 15s/step - accuracy: 0.6605 - loss: 0.6242 - val_accuracy: 0.5000 - val_loss: 0.6943
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 494s 15s/step - accuracy: 0.6630 - loss: 0.6152 - val_accuracy: 0.5000 - val_loss: 0.6959
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 492s 15s/step - accuracy: 0.6765 - loss: 0.6049 - val_accuracy: 0.5520 - val_loss: 0.6912
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 490s 15s/step - accuracy: 0.7060 - loss: 0.5743 - val_accuracy: 0.5150 - val_loss: 0.6919
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 492s 15s/step - accuracy: 0.7115 - loss: 0.5749 - val_accuracy: 0.5000 - val_loss: 0.7114
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 498s 16s/step - accuracy: 0.7145 - loss: 0.5742 - val_accu

이 컨볼루션 신경망은 학습 가능한 매개변수 개수가 721,857개로, 이전 장의 모델이 가진 1,569,089개보다 훨씬 적지만 더 나은 결과를 얻습니다. 그림 9.5는 학습 및 검증 곡선을 보여줍니다.

<img src="https://deeplearningwithpython.io/images/ch09/training-and-validation-xception-acc.967bf65c.png" width="600">

<img src="https://deeplearningwithpython.io/images/ch09/training-and-validation-xception-loss.bc0f5982.png" width="600"><p style="text-align:center">Figure 9.5: Training and validation metrics with a Xception-like architecture</p>

**새로운 모델은 이전 모델의 83.9%에 비해 90.8%의 테스트 정확도를 달성**했습니다. 보시다시피, **아키텍처 모범 사례를 따르는 것은 모델 성능에 즉각적이고 상당한 영향**을 미칩니다!

이 시점에서 성능을 더욱 향상시키려면 아키텍처의 하이퍼파라미터를 체계적으로 튜닝해야 합니다. 이 주제는 18장에서 자세히 다룹니다. 여기서는 이 단계를 다루지 않았으므로 이전 모델의 구성은 순전히 앞에서 설명한 모범 사례를 따랐으며, 모델 크기를 판단할 때는 약간의 직관을 사용했습니다.

### Beyond convolution: Vision Transformers

**컨볼루션 신경망(ConvNet)은 2010년대 중반 이후 컴퓨터 비전 분야를 지배**해 왔지만, **최근에는 비전 트랜스포머(Vision Transformer, 줄여서 ViT)라는 대안 아키텍처와 경쟁하**고 있습니다. 장기적으로는 ViT가 ConvNet을 대체할 가능성이 높지만, 현재로서는 대부분의 경우 ConvNet이 여전히 최선의 선택입니다.

트랜스포머가 무엇인지는 15장에서 자세히 다룰 예정이므로 아직은 모르셔도 됩니다. 간단히 말해, 트랜스포머 아키텍처는 텍스트 처리를 위해 개발된, 근본적으로 순차 처리 아키텍처입니다. 트랜스포머는 순차 처리에 매우 뛰어나기 때문에 이미지 처리에도 사용할 수 있을지에 대한 의문이 제기되었습니다.

ViT는 트랜스포머의 한 유형이므로 마찬가지로 순차 처리를 합니다. 이미지를 1차원 패치 시퀀스로 분할하고, 각 패치를 평면 벡터로 변환한 다음, 벡터 시퀀스를 처리합니다. 트랜스포머 아키텍처 덕분에 ViT는 이미지의 여러 부분 간의 장거리 관계를 포착할 수 있는데, 이는 ConvNet이 때때로 어려움을 겪는 부분입니다.

일반적으로 Transformer는 대규모 데이터셋을 다룰 때 매우 효과적인 선택입니다. 방대한 데이터를 활용하는 데 탁월하기 때문입니다. 하지만 소규모 데이터셋에서는 두 가지 이유로 최적의 선택이 아닙니다. 첫째, ConvNet처럼 공간적 사전 정보를 활용하지 못합니다. ConvNet의 2D 패치 기반 아키텍처는 시각 공간의 지역적 구조에 대한 가정을 더 많이 포함하므로 데이터 효율성이 더 높습니다. 둘째, ViT(Visual Information Technology)가 진가를 발휘하려면 데이터셋 규모가 매우 커야 합니다. ImageNet보다 작은 데이터셋에서는 다루기 어려워집니다.

이미지 인식 분야의 패권 경쟁은 아직 끝나지 않았지만, ViT는 분명 새롭고 흥미로운 장을 열었습니다. 아마도 여러분은 대규모 생성형 이미지 모델에서 이 아키텍처를 접하게 될 것입니다. 이 주제는 17장에서 자세히 다룰 예정입니다. 하지만 소규모 이미지 분류에는 여전히 ConvNet이 최선의 선택입니다.

이것으로 필수적인 ConvNet 아키텍처 모범 사례에 대한 소개를 마치겠습니다. 이러한 원칙들을 숙지하면 다양한 컴퓨터 비전 작업에서 더욱 뛰어난 성능을 발휘하는 모델을 개발할 수 있을 것입니다. 이제 여러분은 숙련된 컴퓨터 비전 전문가가 되는 길에 한 걸음 더 다가섰습니다. 전문성을 더욱 심화시키기 위해 마지막으로 다뤄야 할 중요한 주제가 하나 더 있습니다. 바로 모델이 예측에 도달하는 과정을 해석하는 방법입니다.

### 요약
* 딥러닝 모델의 아키텍처는 당면한 문제의 본질에 대한 핵심 가정을 담고 있습니다.
* 모듈성-계층성-재사용 공식은 딥러닝 모델을 포함한 거의 모든 복잡한 시스템의 아키텍처를 뒷받침합니다.
* 컴퓨터 비전의 주요 아키텍처 패턴에는 잔여 연결, 배치 정규화, 깊이별 분리형 컨볼루션이 있습니다.
* 비전 트랜스포머는 대규모 컴퓨터 비전 작업에서 컨볼루션 신경망(ConvNet)을 대체할 수 있는 유망한 대안입니다.